# 02 – Veri Sızıntısı ve Öznitelik İncelemesi

**Sorumlu:** 3. kişi ·**Branch:** `feature/data-leakage`

## Amaç
Modelin **yalnızca rezervasyon anında bilinen bilgilerle** çalışmasını sağlamak. Bunun için:
1. 32 sütunun ne anlama geldiğini ve ne zaman oluştuğunu belirlemek,
2. Rezervasyondan sonra oluşan ya da değişen (sızıntı yapan) sütunları bulmak,
3. Şüpheli sütunları veriyle test edip gerekçeli karar vermek,
4. Modelde kullanılacak sütunların son listesini çıkarmak.

## Tahmin anı
Ekip kararı: **tahmin, rezervasyon yapıldığı anda yapılır.** Her sütun için sorulan soru şu:
> *"Bu bilgi rezervasyon alındığı anda elimde olur mu?"*

Gerekçe: Veri setinde her rezervasyonun yalnızca son hali var, değişikliklerin tarihi yok. Bu yüzden "girişten önceki bir an" veriden dürüstçe kurulamaz. Künyede de tahmin anı rezervasyon anı olarak tanımlanmıştır.

In [ ]:
import sys
sys.path.append("..")  # notebooks/ klasöründen çalışırken src'yi bulabilmek için

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data import load_raw, TARGET, LEAKAGE_COLS, SUSPECT_COLS

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 50)

df = load_raw()
print(f"Satır: {len(df):,}  Sütun: {df.shape[1]}")
print(f"Genel iptal oranı: {df[TARGET].mean():.3f}")

## 1. Sütun sözlüğü

Tanımlar, veri setini yayımlayan makaleden alınmıştır: Antonio, de Almeida & Nunes (2019), *Hotel booking demand datasets*, Data in Brief 22, 41–49.

- **Oluşma zamanı:** `Rezervasyon anı` = rezervasyon alınırken bilinir · `Sonradan değişebilir` = rezervasyondan sonra güncellenebilir ya da girişte kaydedilir · `Sonuç` = rezervasyonun sonucudur
- **İlk karar:** `Kullan` · `İncele` (bölüm 2'de veriyle test edilecek) · `Çıkar` · `Hedef`

> Bu tablo bir **başlangıç tahmini**. Son kararlar bölüm 2'deki testlerden sonra verilecek.

In [ ]:
sozluk = pd.DataFrame(
    [
        # (sütun, anlamı, tipi, oluşma zamanı, ilk karar, not)
        ("hotel", "Otel türü (Resort Hotel / City Hotel)", "Kategorik", "Rezervasyon anı", "Kullan", ""),
        ("is_canceled", "Rezervasyon iptal edildi mi (1 / 0)", "İkili", "Sonuç", "Hedef", "Tahmin etmeye çalıştığımız değişken"),
        ("lead_time", "Rezervasyonun girildiği gün ile geliş tarihi arasındaki gün sayısı", "Sayısal", "Rezervasyon anı", "Kullan", ""),
        ("arrival_date_year", "Geliş yılı", "Sayısal", "Rezervasyon anı", "İncele", "Sızıntı değil; ama model 2015–2017'yi ezberler, gelecek yıllara genellenmez"),
        ("arrival_date_month", "Geliş ayı", "Kategorik", "Rezervasyon anı", "Kullan", ""),
        ("arrival_date_week_number", "Geliş tarihinin yılın kaçıncı haftası olduğu", "Sayısal", "Rezervasyon anı", "Kullan", "Ay bilgisiyle örtüşüyor, öznitelik aşamasında tartışılabilir"),
        ("arrival_date_day_of_month", "Geliş günü", "Sayısal", "Rezervasyon anı", "Kullan", ""),
        ("stays_in_weekend_nights", "Hafta sonu (Cmt/Paz) kalınacak gece sayısı", "Sayısal", "Rezervasyon anı", "Kullan", ""),
        ("stays_in_week_nights", "Hafta içi (Pzt–Cum) kalınacak gece sayısı", "Sayısal", "Rezervasyon anı", "Kullan", ""),
        ("adults", "Yetişkin sayısı", "Sayısal", "Rezervasyon anı", "Kullan", ""),
        ("children", "Çocuk sayısı", "Sayısal", "Rezervasyon anı", "Kullan", "Eksik değerler 1. kişide"),
        ("babies", "Bebek sayısı", "Sayısal", "Rezervasyon anı", "Kullan", ""),
        ("meal", "Yemek paketi (BB, HB, FB, SC/Undefined)", "Kategorik", "Rezervasyon anı", "Kullan", ""),
        ("country", "Misafirin geldiği ülke (ISO kodu)", "Kategorik", "Sonradan değişebilir", "İncele", "Ülke bazen ancak girişte kesinleşiyor; iptallerde varsayılan değer (PRT) kalmış olabilir"),
        ("market_segment", "Pazar segmenti (Online TA, Groups, Direct...)", "Kategorik", "Rezervasyon anı", "Kullan", ""),
        ("distribution_channel", "Dağıtım kanalı (TA/TO, Direct, Corporate...)", "Kategorik", "Rezervasyon anı", "Kullan", ""),
        ("is_repeated_guest", "Misafir daha önce gelmiş mi (1 / 0)", "İkili", "Rezervasyon anı", "Kullan", ""),
        ("previous_cancellations", "Bu rezervasyondan önce iptal ettiği rezervasyon sayısı", "Sayısal", "Rezervasyon anı", "Kullan", ""),
        ("previous_bookings_not_canceled", "Bu rezervasyondan önce iptal etmediği rezervasyon sayısı", "Sayısal", "Rezervasyon anı", "Kullan", ""),
        ("reserved_room_type", "Rezerve edilen oda tipi (kod)", "Kategorik", "Rezervasyon anı", "Kullan", ""),
        ("assigned_room_type", "Misafire atanan oda tipi (kod)", "Kategorik", "Sonradan değişebilir", "İncele", "Oda genelde girişte atanır"),
        ("booking_changes", "Rezervasyon girildikten sonra, girişe ya da iptale kadar yapılan değişiklik sayısı", "Sayısal", "Sonradan değişebilir", "İncele", "Tanımı gereği rezervasyondan sonra artıyor"),
        ("deposit_type", "Depozito türü (No Deposit, Non Refund, Refundable)", "Kategorik", "Rezervasyon anı", "Kullan", "Non Refund'da iptal oranı tuhaf; 4. kişiyle konuşulacak"),
        ("agent", "Rezervasyonu yapan seyahat acentesinin kimliği", "Kategorik (ID)", "Rezervasyon anı", "Kullan", "Eksik değerler 1. kişide (acente var/yok sütunu)"),
        ("company", "Rezervasyonu yapan / ödeyen şirketin kimliği", "Kategorik (ID)", "Rezervasyon anı", "Kullan", "Büyük çoğunluğu boş; 1. kişi çıkarmayı planlıyor"),
        ("days_in_waiting_list", "Rezervasyonun müşteriye onaylanmadan önce bekleme listesinde kaldığı gün", "Sayısal", "Sonradan değişebilir", "İncele", "Onay anında bilinir; rezervasyon anına sayılıp sayılmayacağı tartışılmalı"),
        ("customer_type", "Müşteri türü (Transient, Contract, Group, Transient-Party)", "Kategorik", "Rezervasyon anı", "Kullan", ""),
        ("adr", "Ortalama günlük ücret (konaklama işlemleri toplamı / gece sayısı)", "Sayısal", "Sonradan değişebilir", "İncele", "Hesaplanma şekli işlemlere dayanıyor; düşük şüphe"),
        ("required_car_parking_spaces", "Talep edilen otopark yeri sayısı", "Sayısal", "Sonradan değişebilir", "İncele", "Girişte kaydediliyor olabilir"),
        ("total_of_special_requests", "Özel istek sayısı (çift kişilik yatak, yüksek kat...)", "Sayısal", "Sonradan değişebilir", "İncele", "Rezervasyondan sonra eklenebilir"),
        ("reservation_status", "Son durum (Canceled, Check-Out, No-Show)", "Kategorik", "Sonuç", "Çıkar", "Hedefin kendisi"),
        ("reservation_status_date", "Son durumun belirlendiği tarih", "Tarih", "Sonuç", "Çıkar", "İptal ya da çıkış tarihi; sonucu ele veriyor"),
    ],
    columns=["sutun", "anlami", "tipi", "olusma_zamani", "ilk_karar", "not"],
)

# Kontrol: sözlükte veri setindeki 32 sütunun hepsi var mı?
assert set(sozluk["sutun"]) == set(df.columns), set(df.columns) ^ set(sozluk["sutun"])
print(sozluk["ilk_karar"].value_counts())
sozluk

## 2. Şüpheli sütunların testi

Yöntem: Her şüpheli sütunun değerlerine göre **iptal oranına** bakılır. Bir değerde iptal oranı %0'a ya da %100'e yapışıyorsa, o sütun büyük olasılıkla sonucu ele veriyordur. Karşılaştırma için genel iptal oranı yaklaşık %37'dir.

Her testten sonra **Yorum** hücresine ne gördüğünü ve kararını yaz.

In [ ]:
def iptal_orani(seri, min_adet=0):
    """Verilen sütunun (ya da serinin) her değeri için kayıt sayısını ve iptal oranını verir."""
    tablo = (
        df.groupby(seri, dropna=False)[TARGET]
        .agg(adet="count", iptal_orani="mean")
        .round({"iptal_orani": 3})
    )
    return tablo[tablo["adet"] >= min_adet]

### 2.1 `assigned_room_type`
Rezerve edilen oda ile atanan oda farklıysa misafir otele gelmiş olabilir. Bu durumda sütun "iptal olmadı" bilgisini taşır.

In [ ]:
oda_degisti = (df["reserved_room_type"] != df["assigned_room_type"]).rename("oda_degisti")
iptal_orani(oda_degisti)

**Yorum:** *(buraya yaz)*

### 2.2 `booking_changes`
Tanımı gereği rezervasyondan sonraki değişiklikleri sayıyor.

In [ ]:
iptal_orani("booking_changes")

**Yorum:** *(buraya yaz)*

### 2.3 `required_car_parking_spaces`

In [ ]:
iptal_orani("required_car_parking_spaces")

**Yorum:** *(buraya yaz)*

### 2.4 `country`
En sık görülen ülkelerin iptal oranlarına bakılır. Özellikle Portekiz'in (`PRT`) oranının diğerlerinden çok farklı olup olmadığı önemli.

In [ ]:
iptal_orani("country", min_adet=1000).sort_values("adet", ascending=False)

**Yorum:** *(buraya yaz)*

### 2.5 `total_of_special_requests`

In [ ]:
iptal_orani("total_of_special_requests")

**Yorum:** *(buraya yaz)*

### 2.6 `days_in_waiting_list`
Değerlerin çoğu 0 olduğu için "bekleme listesinde kaldı mı" diye ikiye ayırarak bakılır.

In [ ]:
bekledi = (df["days_in_waiting_list"] > 0).rename("bekleme_listesinde_kaldi")
iptal_orani(bekledi)

**Yorum:** *(buraya yaz)*

### 2.7 `adr`
İptal edilen ve edilmeyen rezervasyonlarda ücret dağılımı karşılaştırılır. Negatif ve aşırı yüksek değerler 2. kişinin alanında olduğu için grafikte 0–500 aralığı gösterilir.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(data=df[df["adr"].between(0, 500)], x=TARGET, y="adr", ax=ax)
ax.set_xlabel("is_canceled")
ax.set_ylabel("adr")
ax.set_title("İptal durumuna göre ortalama günlük ücret")
plt.show()

df.groupby(TARGET)["adr"].describe().round(1)

**Yorum:** *(buraya yaz)*

### 2.8 `arrival_date_year`
Sızıntı değil, genelleme sorunu. Yıllara göre iptal oranı ve kayıt sayısı:

In [ ]:
iptal_orani("arrival_date_year")

**Yorum:** *(buraya yaz)*

## 3. Kararlar

*(Testler bitince doldurulacak.)* Her sütun için karar ve tek cümlelik gerekçe. Bu tablo `reports/kisitlar_gunlugu.md` dosyasının "Alınan kararlar" bölümüne de eklenecek.

| Sütun | Karar | Gerekçe |
| --- | --- | --- |
| `reservation_status` | Çıkar | Rezervasyonun sonucunu doğrudan veriyor |
| `reservation_status_date` | Çıkar | İptal ya da çıkış tarihi; sonucu ele veriyor |
| | | |

## 4. Modelde kullanılacak sütunlar

*(Kararlardan sonra doldurulacak.)* Son liste `src/data.py` içindeki `LEAKAGE_COLS` ve `SUSPECT_COLS` listelerine de işlenecek.